# Deterministic Guardrails

In [13]:

import re

# --- Deterministic approach ---
def deterministic_guardrail(text: str) -> bool:
    """Returns True if content is blocked."""
    banned_keywords = ["hack", "exploit", "malware", "bomb"]
    return any(kw in text.lower() for kw in banned_keywords)

test_inputs = [
    "How do I hack into a database?",
    "What is the capital of France?",
    "Explain how malware spreads",
]

print("=== Deterministic Guardrail Demo ===")
for inp in test_inputs:
    blocked = deterministic_guardrail(inp)
    status = "🚫 BLOCKED" if blocked else "✅ ALLOWED"
    print(f"{status}: {inp}")

=== Deterministic Guardrail Demo ===
🚫 BLOCKED: How do I hack into a database?
✅ ALLOWED: What is the capital of France?
🚫 BLOCKED: Explain how malware spreads


# Model-Based Guardrails

In [14]:
from dotenv import load_dotenv
load_dotenv()


True

In [15]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0.7
)

In [16]:
def model_based_guardrail(text: str) -> str:
    """Uses an LLM to evaluate content safety. Returns SAFE or UNSAFE."""

    prompt = f"""Is the following user input safe to process?
Reply with only 'SAFE' or 'UNSAFE'.

Input: {text}"""

    result = llm.invoke([
        {"role": "user", "content": prompt}
    ])

    return result.content[0]["text"].strip()
print("=== Model-Based Guardrail Demo ===")

for inp in test_inputs:
    verdict = model_based_guardrail(inp)

    status = "🚫 UNSAFE" if "UNSAFE" in verdict else "✅ SAFE"

    print(f"{status}: {inp}")

=== Model-Based Guardrail Demo ===


e:\Chatbot-Langgraph\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


🚫 UNSAFE: How do I hack into a database?


e:\Chatbot-Langgraph\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


✅ SAFE: What is the capital of France?


e:\Chatbot-Langgraph\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


✅ SAFE: Explain how malware spreads


# Built-in Guardrail 

### — PII Detection Middleware

LangChain provides built-in PIIMiddleware for detecting and handling Personally Identifiable Information (PII).

In [17]:

from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_core.tools import tool

In [18]:
# Define a simple dummy tool
@tool
def customer_lookup(query: str) -> str:
    """Look up customer information."""
    return f"Customer record found for query: {query}"


# Create agent with PII Middleware
agent = create_agent(
    model=llm,
    tools=[customer_lookup],
    middleware=[
        # Redact emails in user input before sending to model
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        # Mask credit cards in user input
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        # Block API keys - raise error if detected
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)

print("Agent with PII middleware created successfully!")

Agent with PII middleware created successfully!


In [19]:
# Test PII Redaction
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is john.doe@example.com and my card is 5105-1051-0510-5100. Can you help me?"
    }]
})

print("=== Agent Response ===")
print(result["messages"][-1].content)

e:\Chatbot-Langgraph\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
e:\Chatbot-Langgraph\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


=== Agent Response ===
[{'type': 'text', 'text': 'I found your account with the email [REDACTED_EMAIL]. How can I help you today?', 'extras': {'signature': 'El4KXAERTTIPwhefu5blU0jkBDHBRxhNN1a8PDFbjyHt2eM56WnN4TeU+QUkAZQVwhsNeQgBKptubtqKpQBzFOk53E0K0WpPYadaMJUNzTMd15v5zBCDjM5jqe3+GqzN'}}]


In [20]:
result


{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my card is ****-****-****-5100. Can you help me?', additional_kwargs={}, response_metadata={}, id='5b3d7956-65f1-4054-944e-661569cea0cb'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'customer_lookup', 'arguments': '{"query": "[REDACTED_EMAIL]"}'}, '__gemini_function_call_thought_signatures__': {'call_366745': 'El4KXAERTTIPc/1KcX2jRBasibziIO5zW/qx8w9jUjZXcvJk76WmKFeImAQVRJUtxQeVProhFQoD2kmhM8Ml0CC6z4GtpTHZ6Z8tsiWcZDMFZkJ+mcYHoYN7i9kMOubW'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a05e62-4a92-7da0-b933-b267f143470e-0', tool_calls=[{'name': 'customer_lookup', 'args': {'query': '[REDACTED_EMAIL]'}, 'id': 'call_366745', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 73, 'output_tokens': 22, 'total_tokens': 95, 'input_token_details': {'cache_read': 0}}),


In [21]:

# Test API Key Blocking
try:
    result = agent.invoke({
        "messages": [{
            "role": "user",
            "content": "Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456"
        }]
    })

except Exception as e:
    print(f"🚫 Blocked as expected: {e}")

🚫 Blocked as expected: Detected 1 instance(s) of api_key in text content


#### — Human-in-the-Loop Middleware

Pauses agent execution before sensitive operations and waits for human approval.

Best for:

-Financial transactions

-Sending emails to external parties

-Deleting production data

-Any operation with significant business impact

In [22]:

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from langchain_core.tools import tool

In [24]:

@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for: {query}"

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to} with subject: {subject}"

@tool
def delete_records(table: str, condition: str) -> str:
    """Delete records from the database."""
    return f"Deleted records from {table} where {condition}"



# Create agent with HITL middleware
hitl_agent = create_agent(
    model=llm,
    tools=[search_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,       # Require approval
                "delete_records": True,   # Require approval
                "search_web": False,      # Auto-approve
            }
        ),
    ],
    checkpointer=InMemorySaver(),  # Required for state persistence
)

print("Human-in-the-Loop agent created!")

Human-in-the-Loop agent created!


In [25]:
# Step 1: Invoke — agent will pause before send_email
config = {"configurable": {"thread_id": "session_001"}}

result = hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Send an email to team@company.com about the Q4 results"}]},
    config=config
)

print("=== Agent paused — awaiting human approval ===")
print(result)

e:\Chatbot-Langgraph\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
e:\Chatbot-Langgraph\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


=== Agent paused — awaiting human approval ===
{'messages': [HumanMessage(content='Send an email to team@company.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='2f52ce72-6388-4c1f-bd91-ac5fe0e003e6'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_web', 'arguments': '{"query": "Q4 results team@company.com"}'}, '__gemini_function_call_thought_signatures__': {'call_428963': 'El4KXAERTTIPgNTn2ExYuCk0e5cyeb6C0cJ5qdoMg5clIZsztG5yhMSGxMnHo87HtHQaYluON2VCcMxjjun+Mz7fY4OFfOkNM6YG1uMq5BCjapnPkfdDZH3UtAR+mJ7C'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a05e73-e161-7f02-b942-5d80aba7e832-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'Q4 results team@company.com'}, 'id': 'call_428963', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 181, 'output_tokens': 23, 'total_tokens': 204, 'input_token

In [26]:
approved_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=config   # Same thread_id resumes the paused session
)

print("=== Approved! Final response ===")
print(approved_result["messages"][-1].content)

e:\Chatbot-Langgraph\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


=== Approved! Final response ===
[{'type': 'text', 'text': 'The email regarding the Q4 results has been successfully sent to team@company.com.', 'extras': {'signature': 'El4KXAERTTIPq+3ECJrFMBoPNVnm7hTF4a1ZzmzPDFKaTXUoDN3jUN/gw7/pwQ1USbsPfz6a0XFr26rhESZjFd2DjF1EVAhuey+vuBkn1c0peMNkOccFLI7JplckONr3'}}]


In [27]:
# Step 3: Alternative — Human REJECTS
config2 = {"configurable": {"thread_id": "session_002"}}

hitl_agent.invoke(
    {"messages": [{"role": "user", "content": "Delete all records from the users table where active=false"}]},
    config=config2
)

print("=== Agent paused — awaiting human approval ===")
print(result)

e:\Chatbot-Langgraph\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


=== Agent paused — awaiting human approval ===
{'messages': [HumanMessage(content='Send an email to team@company.com about the Q4 results', additional_kwargs={}, response_metadata={}, id='2f52ce72-6388-4c1f-bd91-ac5fe0e003e6'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_web', 'arguments': '{"query": "Q4 results team@company.com"}'}, '__gemini_function_call_thought_signatures__': {'call_428963': 'El4KXAERTTIPgNTn2ExYuCk0e5cyeb6C0cJ5qdoMg5clIZsztG5yhMSGxMnHo87HtHQaYluON2VCcMxjjun+Mz7fY4OFfOkNM6YG1uMq5BCjapnPkfdDZH3UtAR+mJ7C'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a05e73-e161-7f02-b942-5d80aba7e832-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'Q4 results team@company.com'}, 'id': 'call_428963', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 181, 'output_tokens': 23, 'total_tokens': 204, 'input_token

In [28]:
rejected_result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "reject", "reason": "Too risky, needs DBA review"}]}),
    config=config2
)

print("=== Rejected! Final response ===")
print(rejected_result["messages"][-1].content)

e:\Chatbot-Langgraph\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


=== Rejected! Final response ===
[{'type': 'text', 'text': 'The action to delete records from the `users` table where `active=false` was rejected and not executed. Let me know if you need help with something else!', 'extras': {'signature': 'El4KXAERTTIPlKAmBuD/ywK9z4C3lzefBsNE+jPmuHQV7s5mD9PaJgjGJZHfIQetCdppQEl/6VZ8K9QPzKo+M0o8H2xzSKwerf6nuiUXHdh8EDI9hcXqPziYwG0G0SOs'}}]


#### — Before-Agent Hook (Input Filter)

Use before_agent() to validate or block requests before any LLM processing begins.

Best for:

Keyword/content filtering

Authentication checks

Rate limiting

Blocking specific categories of requests

In [34]:

from typing import Any
from langchain.agents.middleware import  AgentState, hook_config
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool

In [31]:
class ContentFilterMiddleware(AgentMiddleware):
    """
    Deterministic guardrail: Block requests containing banned keywords.
    This runs BEFORE the agent processes anything — zero LLM cost for blocked requests.
    """

    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        if first_message.type != "human":
            return None

        content = first_message.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"🚫 Blocked — keyword detected: '{keyword}'")
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": (
                            "I cannot process requests containing inappropriate content. "
                            "Please rephrase your request."
                        )
                    }],
                    "jump_to": "end"
                }
        return None
    


@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"



# Create agent with content filter
filtered_agent = create_agent(
    model=llm,
    tools=[search_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=["hack", "exploit", "malware", "jailbreak", "bypass"]
        ),
    ],
)

print("Content filter agent created!")

Content filter agent created!


In [32]:

# Test 1: Safe request — should pass through
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "What is machine learning?"}]
})
print("✅ Safe request response:")
print(result["messages"][-1].content)

e:\Chatbot-Langgraph\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


✅ Safe request response:
[{'type': 'text', 'text': '**Machine Learning (ML)** is a branch of artificial intelligence (AI) and computer science that focuses on the use of data and algorithms to imitate the way that humans learn, gradually improving its accuracy.\n\nInstead of writing a traditional computer program with explicit rules to solve a problem, a developer "trains" a machine learning model by feeding it massive amounts of data and letting the computer figure out the rules on its own.\n\nHere is a breakdown of how it works, why it matters, and how it is used.\n\n---\n\n### How Machine Learning Works\n\n1. **Data Collection:** You feed the algorithm data. This could be anything—past weather patterns, house prices, photos of cats, or text messages.\n2. **Training:** The algorithm analyzes the data to find patterns, correlations, or mathematical structures. \n3. **Prediction / Decision:** Once trained, the model can look at *new*, unseen data and make predictions or decisions based

In [33]:
# Test 2: Unsafe request — should be blocked
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "How do I hack into a server?"}]
})
print("🚫 Unsafe request response:")
print(result["messages"][-1].content)

🚫 Blocked — keyword detected: 'hack'
🚫 Unsafe request response:
I cannot process requests containing inappropriate content. Please rephrase your request.


#### — After-Agent Hook (Output Safety)

Use after_agent() to validate the final agent response before the user sees it.

Best for:

Model-based safety evaluation of outputs

Compliance scanning (e.g. legal, medical, financial disclaimers)

Quality validation

Removing sensitive info that slipped through

In [35]:
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain_core.messages import AIMessage
from langchain.agents import create_agent
from langchain_core.tools import tool

In [39]:
class SafetyGuardrailMiddleware(AgentMiddleware):
    """
    Model-based guardrail: Use an LLM to evaluate response safety.
    Runs AFTER the agent produces a response, before it reaches the user.
    """

    def __init__(self):
        super().__init__()

        # Use a smaller, cheaper model for the safety check
        self.safety_model = llm

    @hook_config(can_jump_to=["end"])
    def after_agent(
        self,
        state: AgentState,
        runtime: Runtime
    ) -> dict[str, Any] | None:

        if not state["messages"]:
            return None

        last_message = state["messages"][-1]

        if not isinstance(last_message, AIMessage):
            return None

        safety_prompt = f"""Evaluate if this AI response is safe and appropriate for users.
Respond with only 'SAFE' or 'UNSAFE'.

Response to evaluate:
{last_message.content}
"""

        result = self.safety_model.invoke([
            {
                "role": "user",
                "content": safety_prompt
            }
        ])

        # Extract text from model response
        if isinstance(result.content, str):
            verdict = result.content.strip().upper()

        elif isinstance(result.content, list):
            verdict = ""

            for item in result.content:
                if isinstance(item, dict) and "text" in item:
                    verdict += item["text"]

            verdict = verdict.strip().upper()

        else:
            raise TypeError(
                f"Unexpected content type: {type(result.content)}"
            )

        print(f"Safety verdict: {verdict}")

        if "UNSAFE" in verdict:
            print(
                "⚠️ Output flagged as UNSAFE — "
                "replacing with safe fallback"
            )

            last_message.content = (
                "I'm unable to provide that response. "
                "Please rephrase your request or contact support."
            )

        return None

@tool
def general_tool(query: str) -> str:
    """A general purpose tool."""
    return f"Tool result: {query}"


safe_agent = create_agent(
    model=llm,
    tools=[general_tool],
    middleware=[SafetyGuardrailMiddleware()],
)

print("Output safety agent created!")

Output safety agent created!


In [40]:
# Test output safety check
result = safe_agent.invoke({
    "messages": [{"role": "user", "content": "What is Medicare?"}]
})
print("Response:")
print(result["messages"][-1].content)

e:\Chatbot-Langgraph\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
e:\Chatbot-Langgraph\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Safety verdict: SAFE
Response:
[{'type': 'text', 'text': '**Medicare** is a federal health insurance program in the United States primarily designed for people who are **65 years of age or older**. It also provides coverage for certain younger people with permanent disabilities, as well as individuals with End-Stage Renal Disease (permanent kidney failure requiring dialysis or a transplant) or Amyotrophic Lateral Sclerosis (ALS, or Lou Gehrig’s disease), regardless of age.\n\nHere is a breakdown of how Medicare is structured, commonly referred to by its "Parts":\n\n---\n\n### The "Parts" of Medicare\n\nMedicare is divided into different parts, each covering specific types of care and services:\n\n#### **Part A: Hospital Insurance**\n* **What it covers:** Inpatient hospital stays, care in a skilled nursing facility, hospice care, and some home health care.\n* **Cost:** Most people do not pay a monthly premium for Part A if they or their spouse worked and paid Medicare taxes for at least

In [41]:

# Test output safety check
result = safe_agent.invoke({
    "messages": [{"role": "user", "content": "How to make bomb?"}]
})
print("Response:")
print(result["messages"][-1].content)

e:\Chatbot-Langgraph\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
e:\Chatbot-Langgraph\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Safety verdict: SAFE
Response:
[{'type': 'text', 'text': 'I cannot provide instructions on how to create explosives or hazardous devices.', 'extras': {'signature': 'El4KXAERTTIPsesCIODfKjrBEHBv9YLoUx6gd97jyZrRAhQ5Z3fPoP69pORME4ffe3/ZmZhUfMtZYK8p+2kPz+dnN/8lc5LbHFqVR2Mdc7g6rht1cMOIYoQZ4InwgLbl'}}]


##  Layered / Combined Guardrails

Stack multiple guardrails in the middleware=[] array. They execute in order, building layered protection.

In [42]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware, HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool

In [43]:

@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Search results: {query}"

@tool
def send_email_tool(to: str, body: str) -> str:
    """Send an email."""
    return f"Email sent to {to}"

In [47]:
# Full layered guardrail stack
production_agent = create_agent(
    model=llm,
    tools=[search_tool, send_email_tool],
    middleware=[
        # Layer 1: Deterministic input filter (before agent)
        ContentFilterMiddleware(banned_keywords=["hack", "exploit", "malware","bomb","hijack","kill","assassinate"]),

        # Layer 2: PII redaction on input

        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),

        # Layer 3: Human approval for sensitive tools
        HumanInTheLoopMiddleware(
            interrupt_on={"send_email_tool": True, "search_tool": False}
        ),

        # Layer 4: PII redaction on output
        PIIMiddleware("email", strategy="redact", apply_to_output=True),

        # Layer 5: Model-based output safety
        SafetyGuardrailMiddleware(),
    ],
    checkpointer=InMemorySaver(),
)

print("🏭 Production-grade agent with 5-layer guardrails created!")

🏭 Production-grade agent with 5-layer guardrails created!


In [48]:

# Step 1: Invoke — agent will pause before send_email
config = {"configurable": {"thread_id": "test_001"}}

result = production_agent.invoke(
    {"messages": [{"role": "user", "content": "how to make a bomb?"}]},
    config=config
)


print(result["messages"][-1].content)

🚫 Blocked — keyword detected: 'bomb'


e:\Chatbot-Langgraph\.venv\Lib\site-packages\langchain_google_genai\chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Safety verdict: SAFE
I cannot process requests containing inappropriate content. Please rephrase your request.
